# Pruebas rapidas
Notebook para probar pandas y matplotlib en el entorno del proyecto.

In [6]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path


In [11]:
Carpeta = r"C:\Users\felix.contreras\Desktop\Gestiones"



carpeta = Path(Carpeta) if "Carpeta" in globals() else Path.cwd()



# Extensiones tabulares soportadas para convertir a DataFrame.

extensiones = {".csv", ".xlsx", ".xls", ".json", ".parquet"}

archivos = sorted([p for p in carpeta.rglob("*") if p.is_file() and p.suffix.lower() in extensiones])



def leer_csv_robusto(path_csv: Path) -> pd.DataFrame:

    """Intenta leer CSV con varios encodings y separadores."""

    encodings = ["utf-8", "utf-8-sig", "cp1252", "latin-1"]

    separadores = [None, ";", ",", "\t", "|"]

    ultimo_error = None



    for enc in encodings:

        for sep in separadores:

            try:

                # sep=None + engine='python' intenta autodetectar delimitador.

                return pd.read_csv(

                    path_csv,

                    encoding=enc,

                    sep=sep,

                    engine="python",

                    on_bad_lines="skip",

                )

            except Exception as e:

                ultimo_error = e



    raise RuntimeError(f"No se pudo leer CSV: {ultimo_error}")



dfs = []

omitidos = []



for archivo in archivos:

    try:

        sufijo = archivo.suffix.lower()

        if sufijo == ".csv":

            tmp = leer_csv_robusto(archivo)

        elif sufijo in {".xlsx", ".xls"}:

            tmp = pd.read_excel(archivo)

        elif sufijo == ".json":

            tmp = pd.read_json(archivo)

        elif sufijo == ".parquet":

            tmp = pd.read_parquet(archivo)

        else:

            continue



        if tmp is None or tmp.empty:

            omitidos.append((archivo.name, "DataFrame vacio"))

            continue



        # Traza de origen para saber de que archivo viene cada fila.

        tmp["origen_archivo"] = archivo.relative_to(carpeta).as_posix()

        dfs.append(tmp)

    except Exception as e:

        omitidos.append((archivo.name, str(e)))



if dfs:

    df_unificado = pd.concat(dfs, ignore_index=True, sort=False)

    print(f"Archivos leidos: {len(dfs)}")

    print(f"Filas totales: {len(df_unificado)}")

    print(f"Columnas totales: {len(df_unificado.columns)}")

    display(df_unificado.head())

else:

    df_unificado = pd.DataFrame()

    print("No se pudieron cargar archivos tabulares compatibles.")



if omitidos:

    print(f"Archivos omitidos por error: {len(omitidos)}")

    display(pd.DataFrame(omitidos, columns=["archivo", "error"]).head(50))


Archivos leidos: 6
Filas totales: 661569
Columnas totales: 30


,fechagestion,horagestion,tiempogestion,tiempollamada,identification,nombrecompleto,cuenta,asesor_gestion,asesor,perfil_historico,...,usuario_mejor_gestion,fecha_mejor_perfil,marca,fecha_pago,valor_pago,acepta_salvamento,crm,monto_inicial,nombre_campana,origen_archivo
0,2026-05-22,12:04:12,00:02:31,00:01:29,1193341064,Maria Yanceli Gutierrez Marcial,181079464-,angie.castro906,NaN,No hubo acuerdo,...,angie.castro906,2026-05-22,30,NaN,NaN,NaN,BSCS,78500.0,Edad 30 Unificado Masivo,reporte_gestiones22y23mayo.csv
1,2026-05-22,09:18:05,00:07:32,00:02:52,1075216023,Miguel Andres Rangel Romero Romero,10650447-,isabel.aristizabal741,NaN,Promesa de Pago,...,isabel.aristizabal741,2026-05-22,30,NaN,NaN,NO,RR,223800.0,Edad 30 Convergente Masivo,reporte_gestiones22y23mayo.csv
2,2026-05-22,09:18:45,00:00:39,00:02:52,1075216023,Miguel Andres Rangel Romero Romero,151781519-,isabel.aristizabal741,NaN,Promesa de pago con descuento,...,isabel.aristizabal741,2026-05-22,castigo,NaN,NaN,NO,BSCS,79200.0,Edad 30 Convergente Masivo,reporte_gestiones22y23mayo.csv
3,2026-05-22,09:21:51,00:01:46,00:01:10,1127216761,Maria Paula Vera Acosta,132926299-,laura.rodriguez535,NaN,Promesa de Pago,...,laura.rodriguez535,2026-05-22,0,NaN,NaN,NO,BSCS,44800.0,Edad 30 Unificado Masivo,reporte_gestiones22y23mayo.csv
4,2026-05-22,09:22:47,00:01:49,00:00:00,1095950293,Eufran Roberto Moya Pedrozo Pedrozo,30995475-,sandra.quiros483,NaN,Promesa de Pago,...,sandra.quiros483,2026-05-22,0_v,NaN,NaN,NO,RR,88600.0,FLP Unificado Masivo,reporte_gestiones22y23mayo.csv


In [16]:
import json

from pathlib import Path



json_path = Path("asesores_catalogo.json")



def guardar_catalogo(catalogo: dict, path: Path = json_path) -> None:

    path.write_text(json.dumps(catalogo, ensure_ascii=False, indent=2), encoding="utf-8")



def cargar_catalogo(path: Path = json_path) -> dict:

    if path.exists():

        return json.loads(path.read_text(encoding="utf-8"))

    # Si no existe, inicia catalogo vacio persistente.

    catalogo = {}

    guardar_catalogo(catalogo, path)

    return catalogo



def normalizar_usuario(usuario: str) -> str:

    return str(usuario).strip().lower().replace(" ", ".")



def crear_asesor(usuario: str, nombre_asesor: str, campo: str, catalogo: dict) -> None:

    u = normalizar_usuario(usuario)

    if u in catalogo:

        raise ValueError(f"El asesor ya existe: {u}")

    catalogo[u] = {"Nombre_Asesor": nombre_asesor.strip(), "Campo": campo.strip()}

    guardar_catalogo(catalogo)



def actualizar_asesor(usuario: str, catalogo: dict, nombre_asesor=None, campo=None) -> None:

    u = normalizar_usuario(usuario)

    if u not in catalogo:

        raise ValueError(f"No existe el asesor: {u}")

    if nombre_asesor is not None:

        catalogo[u]["Nombre_Asesor"] = str(nombre_asesor).strip()

    if campo is not None:

        catalogo[u]["Campo"] = str(campo).strip()

    guardar_catalogo(catalogo)



def borrar_asesor(usuario: str, catalogo: dict) -> None:

    u = normalizar_usuario(usuario)

    if u not in catalogo:

        raise ValueError(f"No existe el asesor: {u}")

    del catalogo[u]

    guardar_catalogo(catalogo)



def ver_catalogo(catalogo: dict) -> pd.DataFrame:

    if not catalogo:

        return pd.DataFrame(columns=["asesor_gestion", "Nombre_Asesor", "Campo"])

    return (

        pd.DataFrame.from_dict(catalogo, orient="index")

        .reset_index()

        .rename(columns={"index": "asesor_gestion"})

        .sort_values(["Campo", "Nombre_Asesor", "asesor_gestion"])

        .reset_index(drop=True)

    )



def aplicar_homologacion(df_base: pd.DataFrame, catalogo: dict) -> pd.DataFrame:

    if "asesor_gestion" not in df_base.columns:

        raise ValueError("La columna 'asesor_gestion' no existe en el DataFrame.")



    base = df_base.copy()

    # Permite reejecutar la celda sin crear sufijos _x/_y.

    cols_a_reemplazar = [c for c in ["Nombre_Asesor", "Campo"] if c in base.columns]

    if cols_a_reemplazar:

        base = base.drop(columns=cols_a_reemplazar)



    mapa_df = ver_catalogo(catalogo)

    salida = base.merge(mapa_df, on="asesor_gestion", how="left")

    salida["Nombre_Asesor"] = salida["Nombre_Asesor"].fillna(salida["asesor_gestion"])

    salida["Campo"] = salida["Campo"].fillna("Pendiente")

    return salida



# Cargar catalogo persistente

asesores_catalogo = cargar_catalogo()

print(f"Catalogo cargado: {len(asesores_catalogo)} asesores")

display(ver_catalogo(asesores_catalogo).head(50))



# Ejemplos de uso (descomenta segun necesites)

# crear_asesor("nuevo.usuario123", "Nuevo Asesor", "Masivo", asesores_catalogo)

# actualizar_asesor("nuevo.usuario123", asesores_catalogo, nombre_asesor="Nuevo Nombre", campo="Empresas")

# borrar_asesor("nuevo.usuario123", asesores_catalogo)



if "df_unificado" in globals() and not df_unificado.empty:

    df_unificado = aplicar_homologacion(df_unificado, asesores_catalogo)

    print(f"Homologacion aplicada sobre {len(df_unificado)} filas")

    faltantes = sorted(set(df_unificado["asesor_gestion"].dropna()) - set(asesores_catalogo.keys()))

    print(f"Asesores sin mapear: {len(faltantes)}")

    if faltantes:

        display(pd.DataFrame({"asesor_gestion_sin_mapear": faltantes}).head(100))


Catalogo cargado: 24 asesores


,asesor_gestion,Nombre_Asesor,Campo
0,leidys.jimenez496,Leidys Jimenez,Empresas
1,mateo.hincapie816,Mateo Hincapie,Empresas
2,ana.cuervo748,Ana Cuervo,Masivo
3,angie.castro906,Angie Castro,Masivo
4,carlos.moya179,Carlos Moya,Masivo
5,darwin.meneses045,Darwin Meneses,Masivo
6,faber.ayala881,Faber Ayala,Masivo
7,isabel.aristizabal741,Isabel Aristizabal,Masivo
8,juan.wilches733,Juan Wilches,Masivo
9,laura.rodriguez535,Laura Rodriguez,Masivo


Homologacion aplicada sobre 661569 filas
Asesores sin mapear: 2


,asesor_gestion_sin_mapear
0,Gestion Masiva
1,Gestion Pagos


In [18]:
df_unificado.info()

<class 'pandas.DataFrame'>
RangeIndex: 661569 entries, 0 to 661568
Data columns (total 32 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   fechagestion           661569 non-null  str    
 1   horagestion            661569 non-null  str    
 2   tiempogestion          68594 non-null   str    
 3   tiempollamada          68594 non-null   str    
 4   identification         661569 non-null  int64  
 5   nombrecompleto         661546 non-null  str    
 6   cuenta                 657951 non-null  str    
 7   asesor_gestion         661569 non-null  str    
 8   asesor                 0 non-null       float64
 9   perfil_historico       661568 non-null  str    
 10  ultimo_perfil_cliente  661443 non-null  str    
 11  valorpromesa           11631 non-null   float64
 12  fechapromesa           11631 non-null   str    
 13  numeromarcado          657951 non-null  object 
 14  intentosmarcacion      661569 non-null  str    

In [20]:
def primera_columna(df_base, candidatas):

    for c in candidatas:

        if c in df_base.columns:

            return c

    return None



def a_hora_hhmmss(serie):

    s = serie.astype("string").str.strip()



    salida = pd.Series(pd.NA, index=serie.index, dtype="string")



    # Intento 1: HH:MM:SS exacto

    dt_hms = pd.to_datetime(s, format="%H:%M:%S", errors="coerce")

    mask_hms = dt_hms.notna()

    salida.loc[mask_hms] = dt_hms.loc[mask_hms].dt.strftime("%H:%M:%S")



    # Intento 2: HH:MM (completa segundos en 00)

    mask_restante = salida.isna()

    if mask_restante.any():

        dt_hm = pd.to_datetime(s.loc[mask_restante], format="%H:%M", errors="coerce")

        mask_hm = dt_hm.notna()

        if mask_hm.any():

            idx_hm = dt_hm.index[mask_hm]

            salida.loc[idx_hm] = dt_hm.loc[idx_hm].dt.strftime("%H:%M:%S")



    # Intento 3: duraciones

    mask_restante = salida.isna()

    if mask_restante.any():

        td = pd.to_timedelta(s.loc[mask_restante], errors="coerce")

        mask_td = td.notna()



        def fmt_td(x):

            if pd.isna(x):

                return pd.NA

            total = int(x.total_seconds())

            if total < 0:

                total = abs(total)

            h = total // 3600

            m = (total % 3600) // 60

            sec = total % 60

            return f"{h:02d}:{m:02d}:{sec:02d}"



        if mask_td.any():

            idx_td = td.index[mask_td]

            salida.loc[idx_td] = td.loc[idx_td].apply(fmt_td).astype("string")



    return salida



if "df_unificado" not in globals() or df_unificado.empty:

    print("Primero ejecuta la celda 3 para construir df_unificado.")

else:

    col_fecha = primera_columna(df_unificado, ["Fecha", "fechagestion", "fecha_gestion"])

    col_hora = primera_columna(df_unificado, ["Hora", "horagestion", "hora_gestion"])

    col_tg = primera_columna(df_unificado, ["Tiempo_Gestion", "tiempogestion", "tiempo_gestion"])

    col_tl = primera_columna(df_unificado, ["Tiempo_Llamada", "tiempollamada", "tiempo_llamada"])

    col_id = primera_columna(df_unificado, ["Identificacion", "identification", "identificacion"])

    col_cuenta = primera_columna(df_unificado, ["Cuenta", "cuenta"])

    col_fp = primera_columna(df_unificado, ["FechaPromesa", "fechapromesa", "fecha_promesa"])



    if col_fecha:

        df_unificado["Fecha"] = pd.to_datetime(df_unificado[col_fecha], errors="coerce").dt.normalize()

    if col_hora:

        df_unificado["Hora"] = a_hora_hhmmss(df_unificado[col_hora])

    if col_tg:

        df_unificado["Tiempo_Gestion"] = a_hora_hhmmss(df_unificado[col_tg])

    if col_tl:

        df_unificado["Tiempo_Llamada"] = a_hora_hhmmss(df_unificado[col_tl])

    if col_id:

        df_unificado["Identificacion"] = df_unificado[col_id].astype("string").str.strip()

    if col_cuenta:

        df_unificado["Cuenta"] = (

            df_unificado[col_cuenta]

            .astype("string")

            .str.replace("-", "", regex=False)

            .str.strip()

        )

    if col_fp:

        df_unificado["FechaPromesa"] = pd.to_datetime(df_unificado[col_fp], errors="coerce").dt.normalize()



    columnas_objetivo = [

        "Fecha",

        "Hora",

        "Tiempo_Gestion",

        "Tiempo_Llamada",

        "Identificacion",

        "Cuenta",

        "FechaPromesa",

    ]



    columnas_disponibles = [c for c in columnas_objetivo if c in df_unificado.columns]

    print("Columnas transformadas:", columnas_disponibles)

    print(df_unificado[columnas_disponibles].dtypes)

    display(df_unificado[columnas_disponibles].head(10))


Columnas transformadas: ['Fecha', 'Hora', 'Tiempo_Gestion', 'Tiempo_Llamada', 'Identificacion', 'Cuenta', 'FechaPromesa']
Fecha             datetime64[us]
Hora                      string
Tiempo_Gestion            string
Tiempo_Llamada            string
Identificacion            string
Cuenta                    string
FechaPromesa      datetime64[us]
dtype: object


,Fecha,Hora,Tiempo_Gestion,Tiempo_Llamada,Identificacion,Cuenta,FechaPromesa
0,2026-05-22,12:04:12,00:02:31,00:01:29,1193341064,181079464,NaT
1,2026-05-22,09:18:05,00:07:32,00:02:52,1075216023,10650447,2026-05-22
2,2026-05-22,09:18:45,00:00:39,00:02:52,1075216023,151781519,2026-05-22
3,2026-05-22,09:21:51,00:01:46,00:01:10,1127216761,132926299,2026-05-22
4,2026-05-22,09:22:47,00:01:49,00:00:00,1095950293,30995475,2026-05-22
5,2026-05-22,09:27:37,00:03:07,00:00:53,7229983,100408373,2026-05-22
6,2026-05-22,09:39:23,00:02:46,00:01:55,27952294,170773835,2026-05-22
7,2026-05-22,09:43:20,00:02:14,00:01:46,1048580410,9876520022361215,2026-05-22
8,2026-05-22,09:52:10,00:04:25,00:00:17,49744237,167523687,2026-05-22
9,2026-05-22,09:55:50,00:06:23,00:05:57,1081398360,167195116,2026-05-22


In [29]:
if "df_unificado" not in globals() or df_unificado.empty:

    print("Primero ejecuta las celdas de carga y transformacion.")

else:

    # Columnas base esperadas por la regla de negocio

    columnas_requeridas = ["Fecha", "Hora", "Cuenta", "Identificacion"]

    faltan = [c for c in columnas_requeridas if c not in df_unificado.columns]



    if faltan:

        print(f"Faltan columnas requeridas: {faltan}")

        print("Ejecuta la celda de transformacion de tipos antes de este paso.")

    else:

        col_asesor = "Nombre_Asesor" if "Nombre_Asesor" in df_unificado.columns else "asesor_gestion"



        # Puedes cambiar esta fecha manualmente (YYYY-MM-DD)

        fecha_filtro = "2026-05-22"

        fecha_filtro = pd.to_datetime(fecha_filtro, errors="coerce")



        if pd.isna(fecha_filtro):

            raise ValueError("fecha_filtro invalida. Usa formato YYYY-MM-DD")



        fecha_filtro = fecha_filtro.normalize()

        base_dia = df_unificado[df_unificado["Fecha"] == fecha_filtro].copy()



        # Limpieza basica de campos clave

        base_dia["Cuenta"] = base_dia["Cuenta"].astype("string").str.strip()

        base_dia.loc[base_dia["Cuenta"].isin(["", "<NA>", "nan", "None"]), "Cuenta"] = pd.NA



        base_dia["Identificacion"] = base_dia["Identificacion"].astype("string").str.strip()

        base_dia.loc[base_dia["Identificacion"].isin(["", "<NA>", "nan", "None"]), "Identificacion"] = pd.NA



        # Llave unica solicitada: Cuenta + Hora por asesor

        base_dia["llave_gestion_unica"] = (

            base_dia["Cuenta"].astype("string").fillna("").str.strip()

            + "|"

            + base_dia["Hora"].astype("string").fillna("").str.strip()

        )



        # Metrica 0: Gest_cuentas (distinct Cuenta)

        resumen_gest_cuentas = (

            base_dia.groupby(col_asesor, dropna=False)["Cuenta"]

            .nunique(dropna=True)

            .reset_index(name="Gest_cuentas")

        )



        # Metrica 1: gestiones unicas (Cuenta + Hora)

        resumen_gestiones = (

            base_dia.groupby(col_asesor, dropna=False)["llave_gestion_unica"]

            .nunique()

            .reset_index(name="gestiones_unicas")

        )



        # Metrica 2: clientes gestionados (Identificacion unica)

        resumen_clientes = (

            base_dia.groupby(col_asesor, dropna=False)["Identificacion"]

            .nunique(dropna=True)

            .reset_index(name="clientes_Gestionados")

        )



        # Perfiles para clasificacion de contacto

        perfiles_contacto_directo = {

            "pago parcial",

            "contesta y cuelga",

            "ya pago",

            "promesa de pago",

            "renuente",

            "llamar luego",

            "no hubo acuerdo",

            "colgo",

            "voluntad de pago",

            "promesa de pago con descuento",

            "no es el encargado del pago",

            "promesa con tercero",

            "dificultad de pago",

            "pago no abonado",

            "reclamacion",

            "recordatorio",

            "encargado renuente",

            "promesa whatsapp",

            "abono",

            "al dia",

        }



        perfiles_contacto_indirecto = {

            "equivocado",

            "mensaje con tercero",

            "tercero no conoce al titular",

            "tercero no toma mensaje",

            "fallecio",

        }



        perfiles_no_contacto = {

            "no contesta",

            "mensaje en buzon",

            "no contacto",

            "ilocalizado",

        }



        perfiles_promesas = {

            "promesa de pago",

            "promesa de pago con descuento",

            "promesa con tercero",

        }



        if "ultimo_perfil_cliente" in base_dia.columns:

            perfil_norm = (

                base_dia["ultimo_perfil_cliente"]

                .astype("string")

                .str.strip()

                .str.lower()

            )



            mask_directo = perfil_norm.isin(perfiles_contacto_directo)

            mask_indirecto = perfil_norm.isin(perfiles_contacto_indirecto)

            mask_no_contacto = perfil_norm.isin(perfiles_no_contacto)

            mask_promesas = perfil_norm.isin(perfiles_promesas)



            resumen_contacto_directo = (

                base_dia.loc[mask_directo]

                .groupby(col_asesor, dropna=False)["Cuenta"]

                .nunique(dropna=True)

                .reset_index(name="contacto_directo")

            )



            resumen_contacto_indirecto = (

                base_dia.loc[mask_indirecto]

                .groupby(col_asesor, dropna=False)["Cuenta"]

                .nunique(dropna=True)

                .reset_index(name="contacto_indirecto")

            )



            resumen_no_contacto = (

                base_dia.loc[mask_no_contacto]

                .groupby(col_asesor, dropna=False)["Cuenta"]

                .nunique(dropna=True)

                .reset_index(name="no_contacto")

            )



            # Metrica 6: Promesas (distinct Cuenta) por perfiles de promesa

            resumen_promesas = (

                base_dia.loc[mask_promesas]

                .groupby(col_asesor, dropna=False)["Cuenta"]

                .nunique(dropna=True)

                .reset_index(name="Promesas")

            )

        else:

            resumen_contacto_directo = pd.DataFrame(columns=[col_asesor, "contacto_directo"])

            resumen_contacto_indirecto = pd.DataFrame(columns=[col_asesor, "contacto_indirecto"])

            resumen_no_contacto = pd.DataFrame(columns=[col_asesor, "no_contacto"])

            resumen_promesas = pd.DataFrame(columns=[col_asesor, "Promesas"])

            print("Advertencia: no existe la columna 'ultimo_perfil_cliente'. Las metricas por perfil quedaran en 0.")



        # Metrica 7: valor_promesa = SUMX(VALUES(Cuenta), MIN(valorpromesa))

        col_valorpromesa = "valorpromesa" if "valorpromesa" in base_dia.columns else None

        if col_valorpromesa is None and "valor_promesa" in base_dia.columns:

            col_valorpromesa = "valor_promesa"



        if col_valorpromesa is not None:

            tmp_valor = base_dia[[col_asesor, "Cuenta", col_valorpromesa]].copy()

            tmp_valor[col_valorpromesa] = pd.to_numeric(tmp_valor[col_valorpromesa], errors="coerce")



            min_por_cuenta = (

                tmp_valor.dropna(subset=["Cuenta"])

                .groupby([col_asesor, "Cuenta"], dropna=False)[col_valorpromesa]

                .min()

                .reset_index(name="min_valor_cuenta")

            )



            resumen_valor_promesa = (

                min_por_cuenta.groupby(col_asesor, dropna=False)["min_valor_cuenta"]

                .sum(min_count=1)

                .reset_index(name="valor_promesa")

            )

        else:

            resumen_valor_promesa = pd.DataFrame(columns=[col_asesor, "valor_promesa"])

            print("Advertencia: no existe columna 'valorpromesa'/'valor_promesa'. valor_promesa quedara en 0.")



        resumen_diario = (

            resumen_gestiones

            .merge(resumen_gest_cuentas, on=col_asesor, how="outer")

            .merge(resumen_clientes, on=col_asesor, how="outer")

            .merge(resumen_contacto_directo, on=col_asesor, how="outer")

            .merge(resumen_contacto_indirecto, on=col_asesor, how="outer")

            .merge(resumen_no_contacto, on=col_asesor, how="outer")

            .merge(resumen_promesas, on=col_asesor, how="outer")

            .merge(resumen_valor_promesa, on=col_asesor, how="outer")

            .fillna(0)

        )



        columnas_enteras = [

            "gestiones_unicas",

            "Gest_cuentas",

            "clientes_Gestionados",

            "contacto_directo",

            "contacto_indirecto",

            "no_contacto",

            "Promesas",

        ]

        for c in columnas_enteras:

            if c in resumen_diario.columns:

                resumen_diario[c] = resumen_diario[c].astype(int)



        if "valor_promesa" in resumen_diario.columns:

            resumen_diario["valor_promesa"] = pd.to_numeric(resumen_diario["valor_promesa"], errors="coerce").fillna(0.0)



        # %_contactabilidad = cuenta_directo / Gest_cuentas

        resumen_diario["%_contactabilidad"] = (

            resumen_diario["contacto_directo"]

            .div(resumen_diario["Gest_cuentas"].replace(0, pd.NA))

            .fillna(0)

            * 100

        )



        # %_Conversion = Promesas / cuenta_directo

        resumen_diario["%_Conversion"] = (

            resumen_diario["Promesas"]

            .div(resumen_diario["contacto_directo"].replace(0, pd.NA))

            .fillna(0)

            * 100

        )



        resumen_diario["%_contactabilidad"] = resumen_diario["%_contactabilidad"].round(2)

        resumen_diario["%_Conversion"] = resumen_diario["%_Conversion"].round(2)



        # Formato moneda: $ 1.000.000 (sin decimales)

        resumen_diario["valor_promesa"] = (

            pd.to_numeric(resumen_diario["valor_promesa"], errors="coerce")

            .fillna(0)

            .round(0)

            .astype("int64")

            .map(lambda x: f"$ {x:,}".replace(",", "."))

        )



        # Formato porcentaje: 50.95%

        resumen_diario["%_contactabilidad"] = resumen_diario["%_contactabilidad"].map(lambda x: f"{x:.2f}%")

        resumen_diario["%_Conversion"] = resumen_diario["%_Conversion"].map(lambda x: f"{x:.2f}%")



        resumen_diario = resumen_diario.sort_values(

            ["gestiones_unicas", "clientes_Gestionados"], ascending=False

        ).reset_index(drop=True)



        print(f"Fecha filtrada: {fecha_filtro.date()}")

        print(f"Registros del dia: {len(base_dia)}")

        print("Resumen por asesor con metricas de contacto, promesas, valor_promesa y porcentajes")

        display(resumen_diario)


Fecha filtrada: 2026-05-22
Registros del dia: 2748
Resumen por asesor con metricas de contacto, promesas, valor_promesa y porcentajes


,Nombre_Asesor,gestiones_unicas,Gest_cuentas,clientes_Gestionados,contacto_directo,contacto_indirecto,no_contacto,Promesas,valor_promesa,%_contactabilidad,%_Conversion
0,Juan Wilches,212,210,210,107,17,86,49,$ 4.511.100,50.95%,45.79%
1,Ruben Baptista,208,207,207,90,29,88,40,$ 2.288.000,43.48%,44.44%
2,Faber Ayala,204,204,203,169,18,17,56,$ 3.242.100,82.84%,33.14%
3,Leidy Rodriguez,192,192,191,152,28,12,38,$ 2.549.200,79.17%,25.00%
4,Paola Lopez,191,191,169,158,26,7,38,$ 2.479.900,82.72%,24.05%
5,Isabel Aristizabal,190,189,185,105,17,67,48,$ 3.223.300,55.56%,45.71%
6,Ana Cuervo,184,184,183,89,79,16,41,$ 2.463.600,48.37%,46.07%
7,Laura Rodriguez,170,168,164,108,22,38,42,$ 2.594.200,64.29%,38.89%
8,Samuel Cardenas,169,169,168,80,22,67,38,$ 2.333.400,47.34%,47.50%
9,Yurani Ramirez,168,168,166,108,31,29,53,$ 3.776.400,64.29%,49.07%


In [ ]:
if "resumen_diario" not in globals() or resumen_diario.empty:

    print("Primero ejecuta la celda 7 para generar resumen_diario.")

else:

    matriz_ui = resumen_diario.copy()

    # Convierte columnas formateadas a numerico para poder colorear dinamicamente.

    matriz_ui["valor_promesa_num"] = (

        matriz_ui["valor_promesa"]

        .astype("string")

        .str.replace("$", "", regex=False)

        .str.replace(" ", "", regex=False)

        .str.replace(".", "", regex=False)

        .pipe(pd.to_numeric, errors="coerce")

        .fillna(0)

    )



    matriz_ui["contactabilidad_num"] = (

        matriz_ui["%_contactabilidad"]

        .astype("string")

        .str.replace("%", "", regex=False)

        .pipe(pd.to_numeric, errors="coerce")

        .fillna(0)

    )



    matriz_ui["conversion_num"] = (

        matriz_ui["%_Conversion"]

        .astype("string")

        .str.replace("%", "", regex=False)

        .pipe(pd.to_numeric, errors="coerce")

        .fillna(0)

    )



    # Sobrescribe con numericos para aplicar formato + iconos en Styler.

    matriz_ui["valor_promesa"] = matriz_ui["valor_promesa_num"]

    matriz_ui["%_contactabilidad"] = matriz_ui["contactabilidad_num"]

    matriz_ui["%_Conversion"] = matriz_ui["conversion_num"]



    def icono_pct(v):

        if v >= 70:

            return "🟢"

        if v >= 50:

            return "🟡"

        return "🔴"



    def icono_money(v):

        if v >= 5000000:

            return "💰💰💰"

        if v >= 2000000:

            return "💰💰"

        return "💰"



    columnas_vista = [

        "Nombre_Asesor",

        "gestiones_unicas",

        "Gest_cuentas",

        "clientes_Gestionados",

        "contacto_directo",

        "contacto_indirecto",

        "no_contacto",

        "Promesas",

        "valor_promesa",

        "%_contactabilidad",

        "%_Conversion",

    ]



    styler = (

        matriz_ui[columnas_vista]

        .style

        .background_gradient(cmap="Blues", subset=["gestiones_unicas", "Gest_cuentas", "clientes_Gestionados"])

        .background_gradient(cmap="Greens", subset=["contacto_directo", "Promesas"])

        .background_gradient(cmap="Oranges", subset=["contacto_indirecto", "no_contacto"])

        .background_gradient(cmap="YlGn", subset=["valor_promesa"])

        .background_gradient(cmap="RdYlGn", subset=["%_contactabilidad", "%_Conversion"])

        .format(

            {

                "valor_promesa": lambda v: f"{icono_money(v)} $ {v:,.0f}".replace(",", "."),

                "%_contactabilidad": lambda v: f"{icono_pct(v)} {v:.2f}%",

                "%_Conversion": lambda v: f"{icono_pct(v)} {v:.2f}%",

            }

        )

    )



    display(styler)


,Nombre_Asesor,gestiones_unicas,Gest_cuentas,clientes_Gestionados,contacto_directo,contacto_indirecto,no_contacto,Promesas,valor_promesa,%_contactabilidad,%_Conversion
0,Juan Wilches,212,210,210,107,17,86,49,💰💰 $ 4.511.100,🟡 50.95%,🔴 45.79%
1,Ruben Baptista,208,207,207,90,29,88,40,💰💰 $ 2.288.000,🔴 43.48%,🔴 44.44%
2,Faber Ayala,204,204,203,169,18,17,56,💰💰 $ 3.242.100,🟢 82.84%,🔴 33.14%
3,Leidy Rodriguez,192,192,191,152,28,12,38,💰💰 $ 2.549.200,🟢 79.17%,🔴 25.00%
4,Paola Lopez,191,191,169,158,26,7,38,💰💰 $ 2.479.900,🟢 82.72%,🔴 24.05%
5,Isabel Aristizabal,190,189,185,105,17,67,48,💰💰 $ 3.223.300,🟡 55.56%,🔴 45.71%
6,Ana Cuervo,184,184,183,89,79,16,41,💰💰 $ 2.463.600,🔴 48.37%,🔴 46.07%
7,Laura Rodriguez,170,168,164,108,22,38,42,💰💰 $ 2.594.200,🟡 64.29%,🔴 38.89%
8,Samuel Cardenas,169,169,168,80,22,67,38,💰💰 $ 2.333.400,🔴 47.34%,🔴 47.50%
9,Yurani Ramirez,168,168,166,108,31,29,53,💰💰 $ 3.776.400,🟡 64.29%,🔴 49.07%
